# 02_运行时与MCP

**来源:**
- [LangChain Docs — Runtime](https://docs.langchain.com/oss/python/deepagents/runtime)
- [LangChain Docs — Context & State](https://docs.langchain.com/oss/python/deepagents/runtime#context--state)
- [LangChain Docs — Store](https://docs.langchain.com/oss/python/deepagents/runtime#store)
- [LangChain Docs — Tool Runtime](https://docs.langchain.com/oss/python/deepagents/tool-runtime)
- [LangChain Docs — MCP](https://docs.langchain.com/oss/python/deepagents/mcp)
- [LangChain Docs — Deep Agents Code MCP](https://docs.langchain.com/docs/deep-agents/code/mcp-tools)
- [MCP 协议规范](https://modelcontextprotocol.io/)

本 Notebook 讲解 Deep Agents 运行时（runtime.context/state/store）、ToolRuntime 以及 MCP（Model Context Protocol）配置。

## 目录

1. [运行时概览](#1-运行时概览)
2. [Runtime Context](#2-runtime-context)
3. [Runtime State](#3-runtime-state)
4. [Runtime Store](#4-runtime-store)
5. [ToolRuntime](#5-toolruntime)
6. [MCP 概述](#6-mcp-概述)
7. [MCP 配置格式](#7-mcp-配置格式)
8. [MCP 工具过滤与安全](#8-mcp-工具过滤与安全)
9. [OAuth 登录](#9-oauth-登录)

**参考链接**
- [Runtime 文档](https://docs.langchain.com/oss/python/deepagents/runtime)
- [ToolRuntime 文档](https://docs.langchain.com/oss/python/deepagents/tool-runtime)
- [MCP 文档](https://docs.langchain.com/oss/python/deepagents/mcp)
- [MCP 协议规范](https://modelcontextprotocol.io/)
- [LangSmith Remote MCP](https://docs.smith.langchain.com/how_to_guides/mcp)

## 1. 运行时概览

Deep Agents 运行时（Runtime）是 Agent 执行的基础设施，包含三个核心概念：

```text
┌─────────────────────────────────────────┐
│             Agent 运行时                   │
│                                         │
│  ┌──────────┐  ┌──────────┐  ┌──────┐   │
│  │ Context  │  │  State   │  │ Store│   │
│  │ (只读)   │  │ (可变)   │  │(持久)│   │
│  └──────────┘  └──────────┘  └──────┘   │
│                                         │
│  ┌──────────────────────────────────┐    │
│  │        ToolRuntime               │    │
│  │  (工具发现、调用、生命周期管理)    │    │
│  └──────────────────────────────────┘    │
└─────────────────────────────────────────┘
```

| 组件 | 作用域 | 持久化 | 说明 |
|------|--------|--------|------|
| **Context** | 请求级别 | ❌ | 只读上下文，包含配置、用户信息等 |
| **State** | 会话级别 | 可选 | 可变状态，通过 checkpointer 持久化 |
| **Store** | 全局级别 | ✅ | 持久化键值存储，跨会话/用户共享 |

## 2. Runtime Context

Runtime Context 是 Agent 执行的只读上下文环境，包含：

- 配置信息（模型、工具、中间件）
- 用户信息（用户 ID、会话 ID）
- 运行时元数据（时间戳、来源）

### 在工具中访问 Context

In [ ]:
# 在工具中通过 langgraph.config 访问运行时上下文
from langgraph.config import get_config


def my_tool_with_context(query: str) -> str:
    """带运行时上下文的工具示例"""
    config = get_config()
    # config 中包含:
    # - configurable.thread_id: 当前会话 ID
    # - configurable.user_id: 用户 ID（如果设置）
    # - metadata: 运行元数据
    thread_id = config.get("configurable", {}).get("thread_id", "unknown")
    return f"[线程: {thread_id}] 查询: {query}"

## 3. Runtime State

State 是 Agent 的可变状态，在 Graph 节点之间传递。在 Deep Agents 中，State 包含 `messages` 列表和可选的自定义字段。

### 通过 Checkpointer 持久化 State

In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver

# 使用 MemorySaver（内存中的检查点）
checkpointer = MemorySaver()

# 或使用 SQLite 持久化检查点
# import sqlite3
# conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
# checkpointer = SqliteSaver(conn)

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    checkpointer=checkpointer,
    system_prompt="你是一个有用的助手。",
)

# 多轮对话 - 使用 thread_id 标识会话
config = {"configurable": {"thread_id": "session-1"}}

# 第一轮
result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "我叫张三"}]},
    config=config,
)

# 第二轮（Agent 记得上一轮内容）
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "我叫什么名字？"}]},
    config=config,
)
print(result2["messages"][-1].content)

### State 管理 API

| 方法 | 说明 |
|------|------|
| `agent.get_state(config)` | 获取当前会话的状态快照 |
| `agent.update_state(config, values)` | 更新当前会话的状态 |
| `agent.get_state_history(config)` | 获取会话的状态历史 |

In [ ]:
# 管理会话状态
# 获取当前状态
# state = agent.get_state(config)
# print(f"消息数: {len(state.values.get('messages', []))}")

# 更新状态（注入系统消息）
# from langchain_core.messages import SystemMessage
# agent.update_state(
#     config,
#     {"messages": [SystemMessage(content="新增指令：用英文回答")]},
#     as_node="model_request",  # 指定在哪个节点后应用
# )

print("状态管理 API 示例 (取消注释后运行)")

## 4. Runtime Store

Store 是全局持久化键值存储，用于在多个会话和用户之间共享数据（如知识库、用户偏好、长期记忆）。

| 特性 | 说明 |
|------|------|
| **作用域** | 全局，跨所有会话 |
| **持久化** | ✅ 自动持久化 |
| **数据结构** | 命名空间下的键值对（namespace/key → value） |
| **典型用途** | 知识库、用户偏好、长期记忆、全局配置 |

In [ ]:
from langgraph.store.memory import InMemoryStore

# 创建内存 Store
store = InMemoryStore()

# 写入 Store
store.put(
    ("user", "zhangsan"),  # namespace
    "preferences",         # key
    {"language": "zh-CN", "theme": "dark"},  # value
)

# 读取 Store
item = store.get(("user", "zhangsan"), "preferences")
print(f"用户偏好: {item.value}")

# 搜索 Store
items = store.search(("user",))
for item in items:
    print(f"{item.key}: {item.value}")

print("\n=== 将 Store 传递给 Agent ===")

# 将 Store 传递给 Agent
agent_with_store = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    store=store,  # 传递 Store 实例
)

## 5. ToolRuntime

ToolRuntime 是 Deep Agents 中工具运行时的核心接口，负责工具的发现、调用和生命周期管理。

In [ ]:
# ToolRuntime 核心概念

from deepagents.runtime import ToolRuntime

# ToolRuntime 负责:
# 1. 工具注册与发现
# 2. 工具调用调度
# 3. 工具执行沙箱化
# 4. 工具结果处理

# 在 create_deep_agent 中，工具通过 tools= 参数注册
# ToolRuntime 自动处理:
# - 工具 Schema 推断
# - 工具调用验证
# - 错误处理与重试
# - 并发控制

print("""
ToolRuntime 负责的工具生命周期:
1. 注册 (Registry) — 将工具函数转换为 BaseTool 对象
2. 验证 (Validation) — 校验参数类型和必需字段
3. 执行 (Execution) — 在沙箱或本地执行工具
4. 结果 (Result) — 格式化工具返回结果
5. 错误恢复 (Error Recovery) — 处理工具调用异常
""")

### ToolRuntime 配置

| 配置项 | 说明 | 默认值 |
|--------|------|--------|
| `max_retries` | 工具调用失败后的最大重试次数 | 3 |
| `execution_timeout` | 工具执行超时时间（秒） | 60 |
| `concurrency_limit` | 工具并发调用限制 | 5 |
| `sandbox` | 沙箱后端，用于隔离执行 | None（本地执行） |

## 6. MCP 概述

MCP（Model Context Protocol）是一种开放协议，允许 Agent 通过外部服务器扩展工具集——文件系统、API、数据库等，而无需修改 Agent 本身。

### 工作原理

```text
┌─────────────┐     MCP 协议     ┌──────────────────┐
│  Deep Agents │ ◄─────────────► │  MCP 服务器      │
│  (Client)   │                  │  - 文件系统      │
│             │                  │  - GitHub        │
│  Agent发现   │                  │  - 数据库        │
│  并调用工具  │                  │  - 自定义API     │
└─────────────┘                  └──────────────────┘
```

### 支持的 MCP 传输方式

| 类型 | 说明 | 适用场景 |
|------|------|---------|
| **stdio** | 通过标准输入/输出运行子进程 | 本地工具、Node.js CLI |
| **SSE** | Server-Sent Events，HTTP 流式 | 远程服务器 |
| **HTTP** | 标准 HTTP 请求 | 简单的 REST API |
| **Streamable HTTP** | 基于 HTTP 的流式传输 | 高性能场景 |

## 7. MCP 配置格式

MCP 服务器通过 JSON 配置文件定义。Deep Agents Code 支持多个自动发现位置。

In [ ]:
# MCP 配置格式

mcp_config_example = {
    "mcpServers": {
        # stdio 服务器（默认）
        "filesystem": {
            "command": "npx",
            "args": [
                "-y",
                "@modelcontextprotocol/server-filesystem",
                "/tmp",
            ],
            "env": {},
        },
        # HTTP 服务器
        "docs-langchain": {
            "type": "http",
            "url": "https://docs.langchain.com/mcp",
        },
        # SSE 服务器
        "remote-api": {
            "type": "sse",
            "url": "https://api.example.com/mcp",
            "headers": {
                "Authorization": "Bearer your-token",
            },
        },
    }
}

import json
print(json.dumps(mcp_config_example, indent=2, ensure_ascii=False))

### 7.1 配置文件自动发现位置

| 优先级 | 位置 | 作用域 |
|--------|------|--------|
| 1 (低) | `~/.deepagents/.mcp.json` | 用户级别，所有项目 |
| 2 | `<项目>/.deepagents/.mcp.json` | 项目级别，`.deepagents` 子目录 |
| 3 (高) | `<项目>/.mcp.json` | 项目级别，根目录（兼容 Claude Code） |

还支持：
- `--mcp-config PATH`：显式指定配置文件路径（最高优先级）
- `--no-mcp`：禁用所有 MCP 加载

## 8. MCP 工具过滤与安全

### 工具过滤

可以限制 MCP 服务器暴露的工具：

In [ ]:
# 工具过滤配置
mcp_config_with_filters = {
    "mcpServers": {
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
            # 只允许读取操作
            "allowedTools": ["read_file", "list_directory"],
        },
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": {"GITHUB_TOKEN": "ghp_..."},
            # 禁止危险操作
            "disabledTools": ["delete_repository"],
        },
    }
}

print("工具过滤配置示例")
print("allowedTools 和 disabledTools 互斥，支持 fnmatch 通配符")

### 8.1 在 SDK 中使用 MCP

Deep Agents SDK 支持通过代码加载 MCP 工具：

In [ ]:
# SDK 中加载 MCP 工具（需要 deepagents MCP 支持）

# from deepagents.mcp import load_mcp_tools

# 从 MCP 服务器加载工具
# mcp_tools = load_mcp_tools(
#     mcp_config=".mcp.json",  # 配置文件路径
# )

# 将 MCP 工具传递给 Agent
# agent = create_deep_agent(
#     model="anthropic:claude-sonnet-4-6",
#     tools=mcp_tools,  # 合并 MCP 工具和自定义工具
# )

print("SDK MCP 工具加载示例 (需要 deepagents MCP 扩展)")

## 9. OAuth 登录

对于需要 OAuth 认证的远程 MCP 服务器，Deep Agents Code 支持 OAuth 登录流程：

In [ ]:
# OAuth MCP 服务器配置
mcp_oauth_config = {
    "mcpServers": {
        "linear": {
            "type": "http",
            "url": "https://mcp.linear.app/mcp",
            "auth": "oauth",  # 启用 OAuth 认证
        },
        # 使用 Bearer Token 的 HTTP 服务器
        "custom-api": {
            "type": "http",
            "url": "https://api.example.com/mcp",
            "headers": {
                "Authorization": "Bearer your-token",
            },
        },
    }
}

print("OAuth MCP 配置示例")
print()
print("运行 OAuth 登录流程:")
print("  dcode mcp login linear")

### MCP 安全最佳实践

| 实践 | 说明 |
|------|------|
| 使用 `allowedTools` | 限制 MCP 服务器暴露的工具范围 |
| 使用 `disabledTools` | 禁用高风险工具（如删除操作） |
| 短生命周期 Secrets | 使用临时令牌而非长期凭据 |
| 项目信任机制 | 确认 MCP 服务器的来源可信 |
| 网络隔离 | 敏感 MCP 服务器应在内网运行 |

---

**参考链接**
- [Runtime 文档](https://docs.langchain.com/oss/python/deepagents/runtime)
- [ToolRuntime 文档](https://docs.langchain.com/oss/python/deepagents/tool-runtime)
- [MCP 文档](https://docs.langchain.com/oss/python/deepagents/mcp)
- [MCP Code 文档](https://docs.langchain.com/docs/deep-agents/code/mcp-tools)
- [MCP 协议规范](https://modelcontextprotocol.io/)
- [LangSmith Remote MCP](https://docs.smith.langchain.com/how_to_guides/mcp)
- [LangGraph Store 文档](https://langchain-ai.github.io/langgraph/concepts/store/)